# $a_1$--$a_2$ prior and posterior contours for every suite

This notebook is the only producer of the $(a_1, a_2)$ coefficient-contour comparison; the per-suite `postfit_physical_parameters.ipynb` notebooks no longer draw it. Edit the configuration cell to choose the suites, priors, and posteriors, then run all cells: one figure is drawn and saved per suite. Each distribution has its 68% credible region filled and its 68% and 95% boundaries outlined; the posteriors are the only solid contours, and every prior takes its own broken pattern so that priors which nearly coincide stay distinguishable where they overlap. The broadest distribution is drawn first, so a compact one is never buried under it. Legend entries follow the standard format, "MINERvA (2026) prior, $k_{\max}=6$" for priors and "Posterior, $k_{\max}=6$" for posteriors, which becomes "Posterior from uniform prior, $k_{\max}=6$" when more than one prior is drawn. Three configuration entries keep that legend readable here: `LABELS` drops the years and the per-entry $k_{\max}$, `NOTE` states the common $k_{\max}=6$ once above the axes, and `COLORS` moves the MINERvA posterior off its prior's color, which a key drawn as both a prior and a posterior would otherwise share.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython.display import display

repo = Path.cwd().resolve()
while repo.name != "axial_mass" and repo != repo.parent:
    repo = repo.parent
if repo.name != "axial_mass":
    raise RuntimeError("Run this notebook from within the axial_mass repository")
helper_dir = repo / "ma_zexp" / "python" / "scripts"
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from postfit_physical_parameters import (
    FIGURE_ROOT, REFERENCE_PRIORS, SPECS, SUITE_DATA_DIRS, load_fit,
    plot_distribution_overlay,
)


## Configuration

`SUITES` lists the fit suites to loop over; each one gets its own figure under `figs/<suite>/comparison_overlays/`. `PRIORS` and `POSTFITS` are independent, so either list may be empty. Standalone prior keys are `deuterium`, `deuterium_k6`, `minerva_k6`, `lqcd_k6`, and `minerva_lqcd_k6`; any z-expansion fit key listed by the next cell can also supply its fitted prior. Posterior keys must exist in every selected suite. Use `deuterium_k6`, rather than native-basis `deuterium`, when comparing with the other $k_{\max}=6$ distributions. `LABELS` and `COLORS` are keyed by fit key for posteriors and by `"<key> prior"` for priors; add a `COLORS` entry whenever a key appears in both lists, since the palette is per source and would hand the prior and the posterior the same color.


In [ ]:
# Every suite gets its own figure. Remove entries to run a subset.
SUITES = [
    "nuwro_fit_results",
    "asimov_fit_results",
    "opendata_fit_results",
]

# Broken contours, one dash pattern each. The deuterium prior is left out:
# the main text barely uses it, and dropping it unclutters the prior cluster.
PRIORS = [
    "minerva_k6",
    "lqcd_k6",
    # "minerva_lqcd_k6",
]

# Solid contours (posteriors). These are loaded from the MCMC output of each suite.
POSTFITS = [
    "minerva_k6_uniform",
    "minerva_k6",
]

# Legend text, overriding the standard "<Source> (year) prior" and
# "Posterior from <source> prior" formats. The years carry no information
# here, and with every distribution at the same k_max, repeating it four
# times only widens the legend: NOTE states it once above the axes instead.
LABELS = {
    "minerva_k6 prior": "MINERvA prior",
    "lqcd_k6 prior": "LQCD prior",
    "minerva_lqcd_k6 prior": "MINERvA + LQCD prior",
    "minerva_k6_uniform": "Posterior, uniform prior",
    "minerva_k6": "Posterior, MINERvA prior",
}

# Contour colors, overriding the per-source palette. A key that appears in
# both PRIORS and POSTFITS would otherwise draw its prior and its posterior
# in one color, so the MINERvA posterior is moved off its prior's blue onto
# the vermillion of the same colorblind-safe palette; the uniform posterior
# keeps the blue it has in the other figures. Keys are the same as LABELS.
COLORS = {
    "minerva_k6": "#D55E00",
}

# Grey line above the axes, carrying what the whole figure shares.
NOTE = r"$k_{\max}=6$    68% and 95% credible regions"

BURN_IN = 0
THIN = 1
N_PRIOR_SAMPLES = 100_000
# SMOOTH is the Gaussian kernel width in bins, so raise it alongside BINS to
# hold the physical smoothing length fixed. This pair is the density estimate
# the figure has always used; a finer grid was tried and changed nothing
# visible in the compact priors.
BINS = 55
SMOOTH = 1.0
FIGSIZE = (7.6, 6.5)
# The legend is two columns, priors on the left and posteriors on the right,
# and LEGEND_HEADROOM opens a band above the contours for it as a fraction of
# the plotted y range. "best" used to drop the legend on top of the contours,
# because there is no free corner large enough for it in any suite; a band of
# its own is the only placement that works for all three. Set LEGEND_HEADROOM
# to 0 and LEGEND_LOC to a corner to put it back inside the contour region.
LEGEND_LOC = "upper center"
LEGEND_HEADROOM = 0.1
SAVE_FIGURE = True
OUTPUT_STEM = "a1_a2_prior_postfit_contours"
SAVE_DPI = 600

In [ ]:
standalone_prior_keys = (
    "deuterium", "deuterium_k6", *REFERENCE_PRIORS.keys(),
)
fit_specs = {spec.key: spec for spec in SPECS if spec.prior is not None}
print("Known suites:", ", ".join(SUITE_DATA_DIRS))
print("Standalone priors:", ", ".join(standalone_prior_keys))
print("Z-expansion fit keys:", ", ".join(fit_specs))

## Validate the selection

The selection is checked once, before any suite is loaded. A selected fit key is loaded once per suite even if both its prior and post-fit distribution are requested. Standalone priors are sampled directly and do not require fit output.


In [ ]:
unknown_postfits = sorted(set(POSTFITS) - set(fit_specs))
unknown_priors = sorted(
    set(PRIORS) - set(fit_specs) - set(standalone_prior_keys)
)
if unknown_postfits:
    raise KeyError(f"Unknown posterior key(s): {unknown_postfits}")
if unknown_priors:
    raise KeyError(f"Unknown prior key(s): {unknown_priors}")
if not SUITES:
    raise ValueError("Select at least one suite")
if not PRIORS and not POSTFITS:
    raise ValueError("Select at least one prior or posterior distribution")

# Fit priors need loading only when they do not also exist as standalone references.
fit_prior_keys = set(PRIORS) - set(standalone_prior_keys)
keys_to_load = sorted(set(POSTFITS) | fit_prior_keys)
selections = (
    [(key, "prior") for key in PRIORS]
    + [(key, "posterior") for key in POSTFITS]
)
print("Fits to load per suite:", ", ".join(keys_to_load) or "none")

## Load, draw, and save one figure per suite

A suite is skipped, with a message, when any requested fit has no unique PROfile ROOT file there.


In [ ]:
for suite in SUITES:
    print(f"\n=== {suite} ===")
    results = {
        key: load_fit(
            fit_specs[key], suite, burn_in=BURN_IN, thin=THIN,
            n_prior=N_PRIOR_SAMPLES,
        )
        for key in keys_to_load
    }
    missing = sorted(key for key, result in results.items() if result is None)
    if missing:
        print(f"Skipping {suite}: no unique PROfile ROOT file found for {missing}")
        continue
    for key, result in results.items():
        print(f"Loaded {key}: {len(result['samples']):,} posterior samples")

    figure = plot_distribution_overlay(
        results, selections, bins=BINS, smooth=SMOOTH,
        n_reference_samples=N_PRIOR_SAMPLES, figsize=FIGSIZE,
        labels=LABELS, colors=COLORS, legend_loc=LEGEND_LOC,
        legend_headroom=LEGEND_HEADROOM, note=NOTE,
    )
    if SAVE_FIGURE:
        output_dir = FIGURE_ROOT / suite / "comparison_overlays"
        output_dir.mkdir(parents=True, exist_ok=True)
        for extension in ("pdf",):
            output_path = output_dir / f"{OUTPUT_STEM}.{extension}"
            figure.savefig(
                output_path, dpi=SAVE_DPI, bbox_inches="tight",
                pad_inches=.03, facecolor="white",
            )
            print("Saved:", output_path)
    display(figure)
    plt.close(figure)